# 🚀 Lumos V3 — GPU Training on T4
> Built from scratch. No Hugging Face models. Real tensors.

**Before you start:** `Runtime → Change runtime type → T4 GPU`

In [ ]:
# ── Step 0: Verify T4 GPU is connected ─────────────────────────────────────
import torch
print(f'GPU Available : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU Name      : {torch.cuda.get_device_name(0)}')
    print(f'GPU RAM       : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('WARNING: No GPU found. Go to Runtime → Change runtime type → T4 GPU')

In [ ]:
# ── Step 1: Clone repo and install dependencies ─────────────────────────────
!git clone https://github.com/Edge-Explorer/Lumos.git
%cd Lumos
!pip install sentencepiece datasets -q
print('All dependencies installed!')

In [ ]:
# ── Step 2: Download & extract clean Sci-Fi stories from HuggingFace ────────
# No CSV upload needed! The WritingPrompts dataset downloads automatically.
!python nano_lumos/data_prep.py

In [ ]:
# ── Step 3: Preview the clean dataset ───────────────────────────────────────
with open('data/sci_fi_nano.txt', 'r') as f:
    lines = f.readlines()
print(f'Total stories: {len(lines)}')
print('\n--- Sample Story ---')
print(lines[0][:500])

In [ ]:
# ── Step 4: Train Lumos V3 on T4 GPU — 50 epochs ────────────────────────────
!python nano_lumos/train.py

In [ ]:
# ── Step 5 (Optional): Monitor GPU in a separate cell ───────────────────────
# Open a new tab and run this while training is happening above
import subprocess, time
for i in range(10):
    r = subprocess.run(
        ['nvidia-smi', '--query-gpu=utilization.gpu,memory.used,memory.total,temperature.gpu',
         '--format=csv,noheader'],
        capture_output=True, text=True
    )
    print(f'[Check {i+1}] GPU Util | Mem Used | Mem Total | Temp → {r.stdout.strip()}')
    time.sleep(30)

In [ ]:
# ── Step 6: Test generation with multiple prompts ──────────────────────────
prompts = [
    "The first human on Mars discovered",
    "A robot woke up alone on an alien planet",
    "In the year 3000 humanity had finally",
]
for p in prompts:
    import subprocess
    result = subprocess.run(['python', 'nano_lumos/generate.py', p], capture_output=True, text=True)
    print(result.stdout)
    print('-' * 60)

In [ ]:
# ── Step 7: Download the trained model ─────────────────────────────────────
from google.colab import files
files.download('lumos_v3.pth')
# Also download the tokenizer
files.download('data/nano_tokenizer.model')